# Lab 5: Nearest Neighbors

Welcome to Lab 5! In this lab, you will build two classifiers from scratch: a 1-nearest-neighbor classifier and a k-nearest-neighbors classifier. You should complete this entire lab so that all tests pass.

We will use a Spotify dataset to predict whether a song is **classical**, **hip-hop**, or **rock**. As in Lecture 7, our two features are `speechiness` and `acousticness`.

**Do not import or use scikit-learn in this lab.** The point is to understand the nearest-neighbor algorithm by writing it yourself.

In [ ]:
in_colab = "google.colab" in str(get_ipython())
if in_colab:
    !pip install otter-grader==6.1.6

from pathlib import Path
import shutil
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

path = 'labs/lab05/build/student'
if in_colab and not Path('data').exists():
    !wget -q -O /content/course.zip https://github.com/dsc-courses/cosmos-ml-cluster-2026/archive/refs/heads/main.zip
    with zipfile.ZipFile('/content/course.zip') as course_zip:
        archive_prefix = f'cosmos-ml-cluster-2026-main/{path}/'
        members = [name for name in course_zip.namelist() if name.startswith(archive_prefix)]
        course_zip.extractall('/content/course-assets', members)
    source_path = Path('/content/course-assets') / archive_prefix
    shutil.copytree(source_path / 'data', 'data', dirs_exist_ok=True)
    shutil.copytree(source_path / 'tests', 'tests', dirs_exist_ok=True)

import otter
grader = otter.Notebook()

plt.style.use('seaborn-v0_8-colorblind')
plt.rcParams['figure.figsize'] = (10, 5)

## The Spotify data 🎵

Each row in `songs` describes one song. The column `genre` is the label we want to predict. We will use all seven numerical audio features:

- `danceability`
- `energy`
- `speechiness`
- `acousticness`
- `instrumentalness`
- `liveness`
- `valence`

In Lecture 7, we used only `speechiness` and `acousticness` because two features can be shown on a scatter plot, which made the nearest-neighbor idea easier to teach. In a real classification task, you would often want to use all of the relevant information available to you. Here, our classifiers will use all seven audio features.

The cells below make a fixed training set and test set. The training songs are the labeled examples a classifier may use; the test songs are held aside until we evaluate the classifier.

In [ ]:
songs = pd.read_csv('data/spotify-three-genres.csv').rename(columns={'track_genre': 'genre'})
songs.head()

## 1. 1-Nearest Neighbor

A **1-nearest-neighbor classifier** predicts the genre of a new song by finding the one labeled training song closest to it and copying that song's genre.

**Question 1.1.** Store a list containing all seven feature-column names, in the order shown above, in `features`. Store the string `'genre'` in `target`. Then create `feature_songs`, a DataFrame containing only the columns in `features`.

In [ ]:
...
feature_songs.head()

In [ ]:
grader.check("q1_1")

Next, run the cell below to create the train and test set you'll use for the remainder of the lab.

In [ ]:
# Create the train/test sets from the feature table.
test_indices = songs.sample(frac=0.2, random_state=42).index
test_songs = songs.loc[test_indices]
training_songs = songs.drop(index=test_indices)
test_features = feature_songs.loc[test_indices]
training_features = feature_songs.drop(index=test_indices)
test_labels = test_songs[target]
training_labels = training_songs[target]

print(f'Training songs: {training_features.shape[0]:,}')
print(f'Test songs: {test_features.shape[0]:,}')

**Question 1.2.** Create a Series named `songs_per_genre` that counts the number of training songs in each genre. Its index should contain the genre names and it should be sorted alphabetically by genre.

Then make a scatter plot of `speechiness` (x-axis) and `acousticness` (y-axis) for the training songs. Color the points according to each song's genre. 

In [ ]:
...

In [ ]:
grader.check("q1_2")

**Question 1.3.** Define a function called `distance`. It should take two Series, `song_a` and `song_b`, each containing numerical feature values, and return their ordinary straight-line distance using every value in the Series.

With $p$ features, the distance is $\sqrt{(a_1-b_1)^2 + (a_2-b_2)^2 + \cdots + (a_p-b_p)^2}$.

**Do not use a `for`-loop for this question.** Use arithmetic on the two Series instead.

In [ ]:
def distance(song_a, song_b):
    """Return the p-dimensional distance between two numerical Series."""
    ...

distance(training_features.iloc[0], training_features.iloc[1])

In [ ]:
grader.check("q1_3")

**Question 1.4.** Define `distances_to_song(new_song, labeled_songs)`. Here, `new_song` is a Series of numerical feature values and `labeled_songs` is a DataFrame containing only numerical feature columns. Return a Series containing the distance from every row of `labeled_songs` to `new_song`. The Series must have the same index as `labeled_songs`. Use arithmetic on DataFrames and Series; do not convert the data to NumPy arrays and back.

In [ ]:
def distances_to_song(new_song, labeled_songs):
    """Return the distance from each numerical feature row to new_song."""
    ...

distances_to_song(test_features.iloc[0], training_features).head()

In [ ]:
grader.check("q1_4")

**Question 1.5.** Define `nearest_song(new_song, labeled_songs)`. It should return the entire numerical feature row of `labeled_songs` that is closest to `new_song`. You may use the function from Question 1.4.

When used on a DataFrame, `.loc` with a single label returns one row as Series by their index label, while `.iloc` selects rows by their integer **position**. For example, `data.loc['song-17']` returns the row whose index label is `'song-17'`, while `data.iloc[0]` returns the first row. The index label returned by `idxmin()` can be used with `.loc`.

In [ ]:
def nearest_song(new_song, labeled_songs):
    """Return the numerical feature row closest to new_song."""
    ...

nearest_song(test_features.iloc[0], training_features)

In [ ]:
grader.check("q1_5")

**Question 1.6.** Define `predict_1nn(new_song, labeled_songs, labels)`. `new_song` and `labeled_songs` contain only numerical features, while `labels` is a Series of genres indexed like `labeled_songs`. Return the label of the closest labeled song as a string. Use `nearest_song` rather than repeating its work.

In [ ]:
def predict_1nn(new_song, labeled_songs, labels):
    """Predict a genre using the closest labeled song."""
    ...

predict_1nn(test_features.iloc[0], training_features, training_labels)

In [ ]:
grader.check("q1_6")

**Question 1.7.** Use `predict_1nn` to predict every song in `test_features`. Store the predictions in a Series named `one_nn_predictions`, with the same index as `test_features`.

In [ ]:
...
one_nn_predictions.head()

In [ ]:
grader.check("q1_7")

**Question 1.8.** Make a DataFrame named `one_nn_results` with the columns `'genre'`, `'predicted_genre'`, and `'correct'`. It should contain the actual genre, the 1-NN prediction, and whether the prediction was correct for every test song. Then assign the proportion of correct predictions to `one_nn_accuracy`.

In [ ]:
...
one_nn_accuracy

In [ ]:
grader.check("q1_8")

**Question 1.9.** Create `one_nn_mistakes`, a DataFrame containing only the rows of `one_nn_results` where the prediction was incorrect. Then store its number of rows in `one_nn_num_mistakes`.

In [ ]:
...

In [ ]:
grader.check("q1_9")

### Listen to some 1-NN mistakes

Run the next cell to inspect a few songs that 1-NN classified incorrectly. The embedded players let you listen to them and compare the actual genre with the prediction.

In [ ]:
from IPython.display import HTML, display

def play_song(track_id, title):
    display(HTML(f'''<p><strong>{title}</strong></p>
    <iframe style="border-radius:12px" src="https://open.spotify.com/embed/track/{track_id}?utm_source=generator"
    width="100%" height="152" frameBorder="0" allowfullscreen=""
    allow="autoplay; clipboard-write; encrypted-media; fullscreen; picture-in-picture" loading="lazy"></iframe>'''))

mistaken_songs = test_songs.loc[one_nn_mistakes.index].copy()
mistaken_songs['predicted_genre'] = one_nn_mistakes['predicted_genre']
songs_to_play = mistaken_songs.head(3)

display(songs_to_play[['track_name', 'artists', 'genre', 'predicted_genre']])

for _, song in songs_to_play.iterrows():
    play_song(
        song['track_id'],
        f"{song['track_name']} — actual: {song['genre']}; predicted: {song['predicted_genre']}"
    )

## 2. k-Nearest Neighbors

A 1-NN classifier lets one training song make the whole decision. A **k-nearest-neighbors (k-NN)** classifier instead finds the $k$ closest training songs and lets them vote. We will use an odd value of $k$ most of the time, but ties can still happen when there are three genres.

**Question 2.1.** Define `k_nearest_songs(new_song, labeled_songs, k)`. It should return the `k` closest rows of `labeled_songs`, sorted from smallest to largest distance. The returned DataFrame must include a new column called `'distance'`.

In [ ]:
def k_nearest_songs(new_song, labeled_songs, k):
    """Return the k numerical feature rows closest to new_song."""
    ...

k_nearest_songs(test_features.iloc[0], training_features, 5)

In [ ]:
grader.check("q2_1")

**Question 2.2.** Define `neighbor_votes(new_song, labeled_songs, labels, k)`. `labeled_songs` contains only numerical features and `labels` contains the corresponding genres. Return a Series whose index contains the genres represented among the `k` nearest songs and whose values are their vote counts.

In [ ]:
def neighbor_votes(new_song, labeled_songs, labels, k):
    """Count the label votes among the k nearest feature rows."""
    ...

neighbor_votes(test_features.iloc[0], training_features, training_labels, 5)

In [ ]:
grader.check("q2_2")

**Question 2.3.** Define `predict_knn(new_song, labeled_songs, labels, k)`. It should return the genre with the most votes among the `k` nearest songs. If two or more genres are tied for the most votes, return the alphabetically first tied genre. For example, a tie between `'hip-hop'` and `'rock'` should return `'hip-hop'`.

In [ ]:
def predict_knn(new_song, labeled_songs, labels, k):
    """Predict a genre using a majority vote among k nearest feature rows."""
    ...

predict_knn(test_features.iloc[0], training_features, training_labels, 5)

In [ ]:
grader.check("q2_3")

**Question 2.4.** Use `predict_knn` with `k=5` to predict every test song. Store the predictions in `five_nn_predictions`. Then create `five_nn_results`, with the same three columns as `one_nn_results`, and assign its proportion of correct predictions to `five_nn_accuracy`.

In [ ]:
...
five_nn_accuracy

In [ ]:
grader.check("q2_4")

**Question 2.5.** Compare the two held-out accuracies. Store `'1-NN'` in `better_model` if `one_nn_accuracy` is larger, `'5-NN'` if `five_nn_accuracy` is larger, and `'tie'` if they are equal.

Then, in the Markdown cell below, explain why 1-NN can be sensitive to an unusual training song.

In [ ]:
...

In [ ]:
grader.check("q2_5")

*Your response here.*

**Question 2.6.** Compare several choices of $k$. Create a DataFrame called `accuracy_by_k`, indexed by the values `1`, `3`, `5`, and `9`, with one column named `'accuracy'`. Each value should be the test accuracy produced by `predict_knn` for that value of $k$. Then make a line plot of the accuracies. The plot is not graded.

You may define a helper function if it makes your solution easier to read.

In [ ]:
...

In [ ]:
grader.check("q2_6")

**Question 2.7.** Assign `best_k` to the value of $k$ in `accuracy_by_k` with the largest test accuracy. If there is a tie, choose the smaller value of $k$.

In [ ]:
...

In [ ]:
grader.check("q2_7")

**Question 2.8.** Which genre is hardest for 5-NN to classify? Create `five_nn_by_genre`, indexed by genre, with one column named `'accuracy'` containing the proportion of correct 5-NN predictions for each actual genre. Then assign `hardest_genre` to the genre with the lowest accuracy.

In [ ]:
...

In [ ]:
grader.check("q2_8")

## Finish Line 🏁

You built 1-NN and k-NN classifiers from scratch. In both cases, the classifier uses labeled examples, so this is **supervised learning**. Using several neighbors can make a classifier less sensitive to any one training song, but a very large value of $k$ can ignore useful local patterns.

In [ ]:
# For your convenience, you can run this cell to run all the tests at once!
grader.check_all()